# 7 – Model Inference
Load the trained classifier adapter, pick any AS-IS process from the eval set, predict which redesign heuristics apply, run the deterministic executor to build the actual TO-BE process, and compare it against the ground-truth redesign.

# Cell 1 – Load base model + trained adapter

In [1]:
import json
import random
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

import bpr_pipeline as bpr

DATA_ROOT = Path("../data").resolve()
ADAPTER_PATH = DATA_ROOT / "qwen_bpr_classifier"
EVAL_DIR = DATA_ROOT / "processed" / "eval"
BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B"

HEURISTIC_NAMES = bpr.APPLICATION_ORDER  # same fixed order used everywhere else
NUM_LABELS = len(HEURISTIC_NAMES)
MAX_LENGTH = 384  # must match what the classifier (notebook 6) was trained with

assert ADAPTER_PATH.exists(), f"Adapter not found at {ADAPTER_PATH} -- train it first in notebook 6"
assert EVAL_DIR.exists(), f"Eval folder not found at {EVAL_DIR} -- run notebook 5 first"

print("CUDA available:", torch.cuda.is_available())

tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_PATH))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    trust_remote_code=True,
)
base_model.config.pad_token_id = tokenizer.pad_token_id

model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
model.eval()
if torch.cuda.is_available():
    model = model.to("cuda")

print("Model + adapter loaded from", ADAPTER_PATH)
print("Device:", next(model.parameters()).device)


CUDA available: True


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model + adapter loaded from C:\Users\yousu\Downloads\SAP\project\data\qwen_bpr_classifier
Device: cuda:0


# Cell 2 – FlowAnalysisService (process metrics: cycle time, cost, ...)

In [2]:
# --- FlowAnalysisService: GraphBuilder / PathEnumerator / calculate_process_metrics ---
# Identical to the hardened version used for scoring in notebook 6.
MAX_COMPOSITE_DEPTH = 3

def _task_times(task: dict):
    proc = task.get("expected_process_time") or 0
    wait = task.get("expected_waiting_time") or 0
    rework = task.get("expected_rework_time") or 0
    return float(proc), float(wait), float(rework)

def _task_cost(task: dict) -> float:
    proc, _wait, rework = _task_times(task)
    d_i_hours = (proc + rework) / 60.0
    cost = 0.0
    for job_task in task.get("jobTasks", []) or []:
        job = job_task.get("job") or {}
        hourly_rate = float(job.get("hourlyRate") or 0)
        alloc_pct = float(job_task.get("time_allocation_percentage") or 0)
        cost += d_i_hours * hourly_rate * (alloc_pct / 100.0)
    return cost

@dataclass
class TaskInfo:
    task_id: int
    order: int
    proc_time: float
    wait_time: float
    rework_time: float
    cost: float
    child_process_id: Optional[int] = None

def _build_task_index(process_json: dict):
    tasks = {}
    for pt in process_json.get("process_task", []) or []:
        task = pt.get("task")
        child_process_id = pt.get("child_process_id")
        if task is None:
            synthetic_id = -(child_process_id or pt.get("process_task_id"))
            tasks[synthetic_id] = TaskInfo(synthetic_id, pt.get("order", 0), 0.0, 0.0, 0.0, 0.0, child_process_id)
            continue
        proc, wait, rework = _task_times(task)
        tasks[task["task_id"]] = TaskInfo(
            task["task_id"], pt.get("order", 0), proc, wait, rework, _task_cost(task), child_process_id
        )
    return tasks

@dataclass
class Node:
    kind: str
    ref: Optional[int] = None
    label: Optional[str] = None
    terminal_after: bool = False

def _normalize_branch_probs(branches):
    raw = [float(b.get("probability") or 0) for b in branches]
    total = sum(raw)
    if total <= 0:
        n = len(branches) or 1
        return [1.0 / n] * len(branches)
    if abs(total - 1.0) < 1e-9:
        return raw
    return [p / total for p in raw]

def _resolve_branch_target(branch):
    if branch.get("target_task_id") is not None:
        return Node(kind="task", ref=branch["target_task_id"], terminal_after=bool(branch.get("connect_to_end")))
    if branch.get("target_gateway_id") is not None:
        return Node(kind="gateway", ref=branch["target_gateway_id"])
    return Node(kind="end", label=branch.get("end_event_name"))

class GraphBuilder:
    def __init__(self, process_json: dict):
        self.tasks = _build_task_index(process_json)
        self.gateways = process_json.get("gateways", []) or []
        self.gateway_by_id = {g["gateway_pk_id"]: g for g in self.gateways}
        self.gateway_by_after_task = {g["after_task_id"]: g for g in self.gateways if g.get("after_task_id") is not None}
        self._ordered_task_ids = sorted(self.tasks.keys(), key=lambda tid: self.tasks[tid].order)

    def find_start_node(self):
        targeted = set()
        for g in self.gateways:
            for b in g.get("branches", []):
                if b.get("target_gateway_id") is not None:
                    targeted.add(b["target_gateway_id"])
        qualifying = [g for g in self.gateways if g.get("after_task_id") is None and g["gateway_pk_id"] not in targeted]
        if qualifying:
            return Node(kind="gateway", ref=qualifying[0]["gateway_pk_id"])
        if self._ordered_task_ids:
            return Node(kind="task", ref=self._ordered_task_ids[0])
        return Node(kind="end", label=None)

    def next_after_task(self, task_id):
        gw = self.gateway_by_after_task.get(task_id)
        if gw is not None:
            return Node(kind="gateway", ref=gw["gateway_pk_id"])
        info = self.tasks.get(task_id)
        if info is None:
            # Dangling reference safety net (see bpr_pipeline._relink_predecessors fix).
            return Node(kind="end", label=None)
        order = info.order
        later = [tid for tid in self._ordered_task_ids if self.tasks[tid].order > order]
        if later:
            return Node(kind="task", ref=later[0])
        return Node(kind="end", label=None)

    def converge_target(self, gateway):
        if gateway.get("converge_at_task_id") is not None:
            return Node(kind="task", ref=gateway["converge_at_task_id"])
        if gateway.get("converge_at_gateway_id") is not None:
            return Node(kind="gateway", ref=gateway["converge_at_gateway_id"])
        if gateway.get("converge_to_end"):
            return Node(kind="end", label=gateway.get("converge_gateway_name"))
        return Node(kind="end", label=None)

@dataclass
class PathResult:
    probability: float
    task_ids: list = field(default_factory=list)
    end_label: Optional[str] = None

class PathEnumerator:
    def __init__(self, graph, child_processes=None, max_paths=5000):
        self.graph = graph
        self.child_processes = child_processes or {}
        self.max_paths = max_paths
        self.warnings = []

    def enumerate(self):
        start = self.graph.find_start_node()
        results = []
        self._walk(start, 1.0, [], frozenset(), 0, results)
        if not results:
            results.append(PathResult(probability=1.0, task_ids=[]))
        return results

    def _walk(self, node, probability, task_ids, visited_gateways, depth, results):
        if len(results) >= self.max_paths:
            self.warnings.append("max_paths limit reached; enumeration truncated")
            return
        if node.kind == "end":
            results.append(PathResult(probability=probability, task_ids=list(task_ids), end_label=node.label))
            return
        if node.kind == "task":
            task_ids = task_ids + [node.ref]
            nxt = Node(kind="end", label=None) if node.terminal_after else self.graph.next_after_task(node.ref)
            self._walk(nxt, probability, task_ids, visited_gateways, depth, results)
            return
        if node.kind == "gateway":
            gateway = self.graph.gateway_by_id.get(node.ref)
            if gateway is None:
                results.append(PathResult(probability=probability, task_ids=list(task_ids)))
                return
            if node.ref in visited_gateways:
                self.warnings.append(f"cyclic reference detected at gateway {node.ref}; loop truncated")
                results.append(PathResult(probability=probability, task_ids=list(task_ids)))
                return
            visited_gateways = visited_gateways | {node.ref}
            gtype = (gateway.get("gateway_type") or "EXCLUSIVE").upper()
            branches = gateway.get("branches", []) or []
            if not branches:
                results.append(PathResult(probability=probability, task_ids=list(task_ids)))
                return
            if gtype == "EXCLUSIVE":
                probs = _normalize_branch_probs(branches)
                for b, p in zip(branches, probs):
                    self._walk(_resolve_branch_target(b), probability * p, task_ids, visited_gateways, depth, results)
                return
            if gtype == "PARALLEL":
                merged = list(task_ids)
                for b in branches:
                    merged += self._collect_branch_tasks(_resolve_branch_target(b), gateway)
                self._walk(self.graph.converge_target(gateway), probability, merged, visited_gateways, depth, results)
                return
            if gtype == "INCLUSIVE":
                n = len(branches)
                probs = [float(b.get("probability") or 0) for b in branches]
                for mask in range(1, 1 << n):
                    subset = [i for i in range(n) if mask & (1 << i)]
                    subset_p = 1.0
                    for i in range(n):
                        subset_p *= probs[i] if i in subset else (1.0 - probs[i])
                    merged = list(task_ids)
                    for i in subset:
                        merged += self._collect_branch_tasks(_resolve_branch_target(branches[i]), gateway)
                    self._walk(self.graph.converge_target(gateway), probability * subset_p, merged, visited_gateways, depth, results)
                return
            probs = _normalize_branch_probs(branches)
            for b, p in zip(branches, probs):
                self._walk(_resolve_branch_target(b), probability * p, task_ids, visited_gateways, depth, results)

    def _collect_branch_tasks(self, node, owning_gateway, _depth=0):
        collected = []
        converge = self.graph.converge_target(owning_gateway)
        cur = node
        while _depth < 500:
            if cur.kind == "end":
                break
            if cur.kind == "task":
                if converge.kind == "task" and cur.ref == converge.ref:
                    break
                collected.append(cur.ref)
                cur = self.graph.next_after_task(cur.ref)
                _depth += 1
                continue
            if cur.kind == "gateway":
                if converge.kind == "gateway" and cur.ref == converge.ref:
                    break
                self.warnings.append(f"nested gateway {cur.ref} inside a parallel/inclusive branch was not expanded")
                break
        return collected

def _path_metrics(path, tasks):
    pt_k = wt_k = rt_k = c_k = 0.0
    for tid in path.task_ids:
        info = tasks.get(tid)
        if info is None:
            continue
        pt_k += info.proc_time
        wt_k += info.wait_time
        rt_k += info.rework_time
        c_k += info.cost
    return {"PT_k": pt_k, "WT_k": wt_k, "RT_k": rt_k, "D_k": pt_k + wt_k + rt_k, "C_k": c_k}

def calculate_process_metrics(process_json, child_processes=None, _depth=0):
    if _depth > MAX_COMPOSITE_DEPTH:
        return {"cycle_time_minutes": 0.0, "labor_cost_per_case": 0.0, "processing_time_minutes": 0.0,
                "waiting_time_minutes": 0.0, "rework_time_minutes": 0.0,
                "cycle_time_efficiency_percent": 0.0, "paths_evaluated": 0,
                "warnings": ["max composite sub-process depth (3) exceeded; returned zeros"]}

    graph = GraphBuilder(process_json)
    enumerator = PathEnumerator(graph, child_processes=child_processes)
    paths = enumerator.enumerate()
    warnings = list(enumerator.warnings)

    for tid, info in graph.tasks.items():
        if info.child_process_id is not None:
            child = (child_processes or {}).get(info.child_process_id)
            if child is not None:
                cr = calculate_process_metrics(child, child_processes=child_processes, _depth=_depth + 1)
                info.proc_time, info.wait_time = cr["processing_time_minutes"], cr["waiting_time_minutes"]
                info.rework_time, info.cost = cr["rework_time_minutes"], cr["labor_cost_per_case"]
            else:
                warnings.append(f"composite sub-process slot references child_process_id={info.child_process_id} "
                                 f"but no matching JSON was supplied; treated as zero-duration/zero-cost")

    e_ct = e_pt = e_wt = e_rt = e_cost = 0.0
    for path in paths:
        m = _path_metrics(path, graph.tasks)
        e_ct += path.probability * m["D_k"]; e_pt += path.probability * m["PT_k"]
        e_wt += path.probability * m["WT_k"]; e_rt += path.probability * m["RT_k"]
        e_cost += path.probability * m["C_k"]

    cte = (e_pt / e_ct * 100.0) if e_ct > 0 else 0.0
    return {"cycle_time_minutes": round(e_ct, 2), "labor_cost_per_case": round(e_cost, 2),
            "processing_time_minutes": round(e_pt, 2), "waiting_time_minutes": round(e_wt, 2),
            "rework_time_minutes": round(e_rt, 2), "cycle_time_efficiency_percent": round(cte, 2),
            "paths_evaluated": len(paths), "warnings": warnings}

print("FlowAnalysisService defined")


FlowAnalysisService defined


# Cell 3 – Executor: apply predicted heuristics deterministically

In [3]:
import copy

# Same executor as notebook 6 -- reuses bpr_pipeline's real apply_* functions directly,
# so the output is guaranteed schema-valid rather than model-generated free text.
_by_name = {name: (hid, apply_fn) for hid, name, qualify, apply_fn in bpr.HEURISTICS}

def redesign_with_predicted_labels(as_is_record: dict, predicted_labels: dict) -> dict:
    working = copy.deepcopy(as_is_record)
    trace = []
    for name in bpr.APPLICATION_ORDER:
        hid, apply_fn = _by_name[name]
        if predicted_labels.get(name, False):
            working, targets, reason = apply_fn(working)
            applied = bool(targets)
            trace.append({
                "heuristicId": hid, "heuristicName": name, "isApplied": applied,
                "taskApplied": targets, "reasonApplied": reason if applied else None,
            })
        else:
            trace.append({
                "heuristicId": hid, "heuristicName": name, "isApplied": False,
                "taskApplied": [], "reasonApplied": None,
            })
    return {"as-is": as_is_record, "to-be": working, "redesignTrace": trace}

print("Executor defined")


Executor defined


# Cell 4 – Prompt builder (must match training exactly)

In [4]:
def record_to_prompt(record):
    process = record["as-is"]
    tasks = [pt["task"]["task_name"] for pt in sorted(process["process_task"], key=lambda x: x["order"])]
    gateways = [f'{g["gateway_type"]} ({g["name"]})' for g in process.get("gateways", [])]

    try:
        measures = bpr.compute_measures(process)
        measure_lines = "\n".join(
            f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}"
            for k, v in measures.items()
        )
    except Exception:
        measure_lines = "(measures unavailable)"

    prompt = f"""### Process Name
{process["process_name"]}

### Tasks
{" -> ".join(tasks)}

### Gateways
{", ".join(gateways) if gateways else "None"}

### Process Measures
{measure_lines}
"""
    return prompt.strip()

print("Prompt builder defined")


Prompt builder defined


# Cell 5 – Browse available processes in the eval set

In [5]:
eval_files = sorted(EVAL_DIR.glob("*.json"))
print(f"{len(eval_files)} processes available in {EVAL_DIR}\n")

print("A random sample of process codes you can pick in Cell 6:")
for fp in random.sample(eval_files, min(10, len(eval_files))):
    record = json.loads(fp.read_text(encoding="utf-8"))
    name = record["as-is"]["process_name"]
    task_count = len(record["as-is"]["process_task"])
    print(f'  {fp.stem:16s}  "{name}"  ({task_count} tasks)')

22084 processes available in C:\Users\yousu\Downloads\SAP\project\data\processed\eval

A random sample of process codes you can pick in Cell 6:
  SYN-P-233128      "timer1"  (3 tasks)
  SYN-P-193258      "W2-P4A(ORDER-TO-CASH)"  (6 tasks)
  SYN-P-145522      "Phase 3"  (4 tasks)
  SYN-P-172988      "lol"  (18 tasks)
  SYN-P-270463      "2.3. Review Acceptability (2ndAssignment)"  (4 tasks)
  SYN-P-190922      "Functional requirements"  (8 tasks)
  SYN-P-124890      "7.5.2 Manage and administer benefits"  (4 tasks)
  SYN-P-269343      "12345678"  (3 tasks)
  SYN-P-237232      "University Admission To-Be 2"  (12 tasks)
  SYN-P-219597      "Loan Application Process"  (4 tasks)


# Cell 6 – Select a process

In [6]:
# Set this to a specific process code from Cell 5's list (e.g. "SYN-P-123456"),
# or leave as None to pick a random one from the eval set each time this cell runs.
PROCESS_CODE = None

if PROCESS_CODE:
    fp = EVAL_DIR / f"{PROCESS_CODE}.json"
    if not fp.exists():
        raise FileNotFoundError(f"{fp} not found -- check the process code against Cell 5's list")
else:
    fp = random.choice(eval_files)

record = json.loads(fp.read_text(encoding="utf-8"))

print("Selected process:", fp.stem)
print("Process name:    ", record["as-is"]["process_name"])
print("AS-IS task count:", len(record["as-is"]["process_task"]))
print("AS-IS gateways:  ", len(record["as-is"].get("gateways", [])))


Selected process: SYN-P-163228
Process name:     Check for availability of materials
AS-IS task count: 4
AS-IS gateways:   2


# Cell 7 – Predict which heuristics apply

In [7]:
@torch.no_grad()
def predict_labels(record):
    prompt = record_to_prompt(record)
    inputs = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH,
                        padding=True, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits
    probs = torch.sigmoid(logits).float().cpu().numpy()[0]
    labels = {name: bool(probs[i] >= 0.5) for i, name in enumerate(HEURISTIC_NAMES)}
    prob_by_name = dict(zip(HEURISTIC_NAMES, probs.tolist()))
    return labels, prob_by_name

predicted_labels, probabilities = predict_labels(record)
ground_truth_applied = {t["heuristicName"]: t["isApplied"] for t in record["redesignTrace"]}

print(f"{'Heuristic':24s} {'Pred':>5s} {'GT':>4s} {'Prob':>7s}   Match?")
for name in HEURISTIC_NAMES:
    pred = predicted_labels[name]
    gt = ground_truth_applied.get(name, False)
    match = "OK" if pred == gt else "MISMATCH"
    print(f"{name:24s} {str(pred):>5s} {str(gt):>4s} {probabilities[name]:7.3f}   {match}")


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Heuristic                 Pred   GT    Prob   Match?
task_elimination          True True   1.000   OK
task_composition         False True   0.020   MISMATCH
resequencing             False False   0.000   OK
knock_out                False False   0.008   OK
parallelism              False False   0.000   OK
case_based_work          False False   0.000   OK
numerical_involvement     True True   1.000   OK
task_automation           True False   0.988   MISMATCH
trusted_party             True False   0.625   MISMATCH
extra_resources           True True   0.500   OK


# Cell 8 – Run the executor to build the actual TO-BE process

In [8]:
combined = redesign_with_predicted_labels(record["as-is"], predicted_labels)
predicted_to_be = combined["to-be"]
predicted_trace = combined["redesignTrace"]

problems = bpr.validate_record(predicted_to_be)
if problems:
    print("WARNING: executed to-be FAILED schema validation:")
    for p in problems:
        print("  -", p)
else:
    print("Executed to-be passed schema validation.")


Executed to-be passed schema validation.


# Cell 9 – Compare AS-IS vs predicted TO-BE vs ground-truth TO-BE

In [9]:
def pct_reduction(before, after):
    return 0.0 if before == 0 else (before - after) / before * 100

as_is_metrics = calculate_process_metrics(record["as-is"])
gt_metrics = calculate_process_metrics(record["to-be"])
pred_metrics = calculate_process_metrics(predicted_to_be)

print(f"{'Metric':28s} {'AS-IS':>12s} {'TO-BE (model)':>15s} {'TO-BE (ground truth)':>22s}")
rows = [
    ("cycle_time_minutes", "Cycle time (min)"),
    ("labor_cost_per_case", "Cost per case ($)"),
    ("cycle_time_efficiency_percent", "Cycle efficiency (%)"),
]
for key, label in rows:
    print(f"{label:28s} {as_is_metrics[key]:>12.2f} {pred_metrics[key]:>15.2f} {gt_metrics[key]:>22.2f}")

print()
ct_pred_reduction = pct_reduction(as_is_metrics["cycle_time_minutes"], pred_metrics["cycle_time_minutes"])
ct_gt_reduction = pct_reduction(as_is_metrics["cycle_time_minutes"], gt_metrics["cycle_time_minutes"])
cost_pred_reduction = pct_reduction(as_is_metrics["labor_cost_per_case"], pred_metrics["labor_cost_per_case"])
cost_gt_reduction = pct_reduction(as_is_metrics["labor_cost_per_case"], gt_metrics["labor_cost_per_case"])

print(f"Cycle time reduction -- model: {ct_pred_reduction:5.1f}%   ground truth: {ct_gt_reduction:5.1f}%")
print(f"Cost reduction       -- model: {cost_pred_reduction:5.1f}%   ground truth: {cost_gt_reduction:5.1f}%")


Metric                              AS-IS   TO-BE (model)   TO-BE (ground truth)
Cycle time (min)                   379.95          245.31                 372.00
Cost per case ($)                   48.82           18.95                  82.50
Cycle efficiency (%)                87.46           80.76                  81.45

Cycle time reduction -- model:  35.4%   ground truth:   2.1%
Cost reduction       -- model:  61.2%   ground truth: -69.0%


# Cell 10 – Human-readable redesign trace

In [10]:
process_name = record["as-is"]["process_name"]
print(f'Redesign trace for "{process_name}" ({fp.stem})\n')

for t in predicted_trace:
    status = "APPLIED" if t["isApplied"] else "not applied"
    print(f"[{t['heuristicId']:>2}] {t['heuristicName']:24s} -- {status}")
    if t["isApplied"]:
        print(f"     reason: {t['reasonApplied']}")
        for target in t["taskApplied"]:
            print(f"     affected: {target}")
    print()

Redesign trace for "Check for availability of materials" (SYN-P-163228)

[ 2] task_elimination         -- APPLIED
     reason: Task 'Check whether missing quantities can be compensated by storage' is a control/check task adding no direct customer value; removed per task elimination heuristic.
     affected: {'task_id': 2, 'task_name': 'Check whether missing quantities can be compensated by storage'}

[ 4] task_composition         -- not applied

[ 8] resequencing             -- not applied

[ 7] knock_out                -- not applied

[ 1] parallelism              -- not applied

[ 5] case_based_work          -- not applied

[ 6] numerical_involvement    -- APPLIED
     reason: Task 'Search for alternative supplier' had multiple role assignments; consolidated to a single owning role.
     affected: {'task_id': 3, 'task_name': 'Search for alternative supplier'}

[ 3] task_automation          -- APPLIED
     reason: Task 'Send information to production planning.' is a manual communicati

# Cell 11 – Save & print the final JSON

In [11]:
output = {
    "process_code": fp.stem,
    "as-is": record["as-is"],
    "to-be": predicted_to_be,
    "redesignTrace": predicted_trace,
}

OUT_DIR = DATA_ROOT / "inference_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / f"{fp.stem}_predicted.json"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Saved full predicted redesign to {OUT_PATH}\n")
print(json.dumps(output, indent=2, ensure_ascii=False))


Saved full predicted redesign to C:\Users\yousu\Downloads\SAP\project\data\inference_output\SYN-P-163228_predicted.json

{
  "process_code": "SYN-P-163228",
  "as-is": {
    "process_id": 163228,
    "company_id": 1063228,
    "created_at": "2021-02-01 09:30:26",
    "updated_at": "2021-02-01 09:30:26",
    "capacity_requirement_minutes": 450,
    "parent_process_id": null,
    "parent_task_id": null,
    "process_code": "SYN-P-163228",
    "process_name": "Check for availability of materials",
    "process_overview": "<p>No description provided in source data.</p>",
    "process_category_id": 1,
    "process_status_id": 1,
    "process_version": 0,
    "bpmn_xml": "<?xml version='1.0' encoding='utf-8'?>\n<bpmn:definitions xmlns:bpmn=\"http://www.omg.org/spec/BPMN/20100524/MODEL\" id=\"Definitions_SYN-P-163228\" targetNamespace=\"http://synthetic.local/bpmn\"><bpmn:process id=\"Process_SYN-P-163228\" name=\"Check for availability of materials\" isExecutable=\"false\"><bpmn:startEvent i